# Vantage Vision: train the obstacle model on a free Google GPU

Runs the whole pipeline on Colab's free T4 GPU, about **2–3 hours** in total:

| step | what happens | time |
|---|---|---|
| 1 | download ROD street obstacles + COCO indoor rooms, merge into 29 Vantage classes | ~20 min |
| 2 | train YOLO11n at full 640 px resolution (40 epochs) | ~2–2.5 h |
| 3 | test on unseen photos and compare with the stock camera model | ~5 min |
| 4 | export for the Raspberry Pi AI Camera (IMX500) | ~10 min |
| 5 | download one zip that drops straight into the laptop / Pi folder | — |

**Before you start**
1. `Runtime → Change runtime type → T4 GPU → Save`.
2. Run the cells **one at a time, top to bottom** (Shift+Enter).

Progress is saved to your **Google Drive**. If Colab disconnects, reconnect, run cells 0–1 again, then run the *Train* cell: it continues from where it stopped.

## 0. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU - set Runtime > Change runtime type > T4 GPU"

## 1. Connect Google Drive and get the code
* Allow access to Google Drive when asked. The project is kept in `My Drive/Vantage-Training`.
* The repository is private, so the first time you need a **GitHub token** with read access:
  github.com → your picture → Settings → Developer settings → Personal access tokens → Fine-grained tokens →
  Generate new token → Repository access: *Vantage-Training* → Permissions: Contents *Read-only*.
* Optional: a free **Hugging Face token** (huggingface.co → Settings → Access Tokens → *Read*) makes the dataset
  download several times faster. Press Enter to skip.

In [ ]:
import os, getpass, subprocess
from google.colab import drive
drive.mount("/content/drive")

BRANCH = "claude/vantage-core-script-srhw18"
REPO = "/content/drive/MyDrive/Vantage-Training"
if not os.path.exists(REPO):
    tok = getpass.getpass("GitHub token: ").strip()
    url = f"https://{tok}@github.com/vantagehalo-bot/Vantage-Training.git"
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=False)
%cd {REPO}

if not os.environ.get("HF_TOKEN"):
    hf = getpass.getpass("Hugging Face token (optional, Enter to skip): ").strip()
    if hf:
        os.environ["HF_TOKEN"] = hf

!pip -q install ultralytics huggingface_hub pyarrow
DATA = "/content/vantage_data"      # fast local disk; rebuilt automatically after a disconnect

## 2. Build the dataset
* **ROD**: 24k street photos from a pedestrian's eye level (stairs, manholes, poles, cones, crossings ...). MIT licence.
* **COCO 2017**: only photos containing indoor obstacles (tables, sofas, beds, TVs, suitcases ...). CC BY 4.0.

Skipped automatically if it already exists in this Colab session.

In [ ]:
if not os.path.exists(f"{DATA}/data.yaml"):
    !python vision/build_dataset.py --out {DATA} --raw /content/raw --coco-shards 6
else:
    print("Dataset already built:", DATA)

In [ ]:
# A look at some training pictures with their labels
import random, glob, cv2, sys, matplotlib.pyplot as plt
sys.path.insert(0, "vision")
import taxonomy as tx
from draw import draw_detections
fig, axs = plt.subplots(2, 3, figsize=(18, 9))
for ax, p in zip(axs.flat, random.sample(glob.glob(f"{DATA}/images/train/*.jpg"), 6)):
    lbl = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    dets = []
    for ln in open(lbl).read().splitlines():
        c, x, y, w, h = ln.split(); x, y, w, h = map(float, (x, y, w, h))
        dets.append((tx.NAMES[int(c)], 1.0, (x - w/2, y - h/2, x + w/2, y + h/2)))
    ax.imshow(draw_detections(cv2.imread(p), dets)[:, :, ::-1]); ax.axis("off")
plt.tight_layout()

## 3. Train
YOLO11n at 640 px, starting from COCO-pretrained weights. Each epoch takes ~3 minutes on a T4.
Watch `mAP50` (accuracy) rise in the printout. Weights are saved to Drive after every epoch.

**If Colab disconnected:** re-run cells 0 and 1 (and 2, which rebuilds the dataset), then this cell resumes.

In [ ]:
last = f"{REPO}/vision/runs/vantage_gpu/weights/last.pt"
if os.path.exists(last):
    print("Resuming from", last)
    !python vision/train.py --name vantage_gpu --resume
else:
    !python vision/train.py --data {DATA}/data.yaml --epochs 40 --imgsz 640 --batch 32 --device 0 --workers 4 --name vantage_gpu

In [ ]:
from IPython.display import Image, display
display(Image("vision/runs/vantage_gpu/results.png", width=1000))
display(Image("vision/runs/vantage_gpu/confusion_matrix_normalized.png", width=800))

## 4. Test on photos the model has never seen
Compares our model with the stock COCO model that ships with the AI camera, and builds the expo page.

In [ ]:
!python vision/evaluate.py --weights vision/models/vantage_gpu.pt --data {DATA}/data.yaml --n 800
!cp {DATA}/stats.json vision/reports/stats.json
!cp vision/runs/vantage_gpu/results.csv vision/reports/results.csv
!python vision/expo_page.py --run vision/runs/vantage_gpu --dataset {DATA} --note "Trained for 40 epochs at 640 px on a free Colab T4 GPU."
display(Image("vision/reports/recall_chart.png", width=800))
for p in sorted(glob.glob("vision/reports/examples/*.jpg"))[:4]:
    display(Image(p, width=1100))

## 5. Export for the Raspberry Pi AI Camera (IMX500)
Converts the model to 8-bit numbers and compiles it with Sony's toolchain (installs Java + the converter the first time).

In [ ]:
import sys
!cp vision/models/vantage_gpu.pt vision/models/vantage.pt
if sys.version_info >= (3, 13):
    print("This Colab runtime uses Python", sys.version.split()[0], "- Sony's IMX500 converter needs 3.10-3.12.")
    print("The trained model is still fine for the laptop demo. Export for the Pi later with: python vision/export_imx.py")
else:
    !pip -q install "pulp<3"  # Sony's quantiser breaks on PuLP 3+
    !python vision/export_imx.py --weights vision/models/vantage.pt --data {DATA}/data.yaml --imgsz 640


## 6. Download everything
`vantage_model_bundle.zip` contains `models/` (the trained model + IMX500 package) and `reports/` (results, charts, expo page).

**On the laptop:** unzip it into the `Vantage-Training\vision` folder (replace files when asked), then run `python vision/demo_app.py`.

**On the Pi:**
```bash
cd ~/Vantage-Training && unzip -o vantage_model_bundle.zip -d vision/
sudo apt install imx500-all
imx500-package -i vision/models/vantage_imx_model/packerOut.zip -o vision/models/vantage_rpk
python3 vantage_core.py --model vision/models/vantage_rpk/network.rpk \
    --labels vision/models/vantage_imx_model/labels.txt --bbox-normalization --bbox-order xy
```

In [ ]:
!cd vision && rm -f /content/vantage_model_bundle.zip && zip -qr /content/vantage_model_bundle.zip models/vantage.pt reports -x "reports/val/*" && (test -d models/vantage_imx_model && zip -qr /content/vantage_model_bundle.zip models/vantage_imx_model || true)
!cp /content/vantage_model_bundle.zip {REPO}/
from google.colab import files
files.download("/content/vantage_model_bundle.zip")
print("Also saved to Google Drive: My Drive/Vantage-Training/vantage_model_bundle.zip")

## 7. (Optional) Try the demo right here
Creates a temporary public link (valid ~72 h) you can open on any phone or laptop. Stop it with the ■ button.

In [ ]:
!pip -q install gradio
!python vision/demo_app.py --share --run vision/runs/vantage_gpu